# Tool 호출 정확도 개선

Agent가 올바른 Tool을 선택하고, 올바른 파라미터로 호출하는 것은 **Tool 설명(description)의 품질**에 달려 있다.

이 실습에서는 같은 기능의 Tool을 **나쁜 설명 vs 좋은 설명**으로 비교하여 정확도 차이를 확인한다.

### 학습 목표

1. Tool 설명이 호출 정확도에 미치는 영향을 체감한다
2. Tool 호출 실패 4가지 유형을 식별한다
3. 파라미터 Validation 레이어를 추가하는 방법을 익힌다
4. Langfuse로 Tool 호출 패턴을 분석한다

---

In [10]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langchain.agents import create_agent
from langfuse.langchain import CallbackHandler

## 1. 나쁜 Tool 설명의 문제

Tool 이름과 설명이 모호하면 LLM이 엉뚱한 Tool을 선택하거나, 아예 Tool을 쓰지 않는다.

> **Tool 호출 실패 4가지 유형**:
> 1. **Wrong Tool Selected** — 엉뚱한 Tool 선택
> 2. **Wrong Parameters** — 올바른 Tool, 잘못된 파라미터
> 3. **Missing Validation** — 파라미터 타입/범위 오류
> 4. **Schema Mismatch** — API 스키마 변경 미반영

In [11]:
# ── 나쁜 Tool 정의: 모호한 이름과 설명 ──

@tool
def search(query: str) -> str:
    """검색한다."""
    return f"[사내 문서] '{query}' 검색 결과: 재택근무는 주 3회 가능, 팀장 승인 필요"


@tool
def lookup(query: str) -> str:
    """조회한다."""
    return f"[인사 정보] {query}: 개발팀 시니어 엔지니어, 내선 1234"


@tool
def get_info(query: str) -> str:
    """정보를 가져온다."""
    return f"[일정] {query}: 10시 팀회의, 14시 회의실A 예약됨"


bad_tools = [search, lookup, get_info]
print("나쁜 Tool 목록:")
for t in bad_tools:
    print(f"  - {t.name}: {t.description}")

나쁜 Tool 목록:
  - search: 검색한다.
  - lookup: 조회한다.
  - get_info: 정보를 가져온다.


In [12]:
# ── 테스트 질문 ──
test_questions = [
    "재택근무 신청은 어떻게 하나요?",         # → 사내 문서 검색
    "김철수 대리 내선번호 알려주세요",          # → 인사 정보 조회
    "내일 오후 회의실 빈 시간 알려주세요",      # → 일정 조회
    "오늘 서울 날씨 어때요?",                  # → Tool 사용 불필요
]

# ── 나쁜 Tool로 에이전트 실행 ──
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
bad_agent = create_agent(model=llm, tools=bad_tools)
handler = CallbackHandler()

bad_results = []
print("🔴 나쁜 Tool 설명으로 실행:\n")
for q in test_questions:
    result = bad_agent.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["bad-tools", "tool-tuning-lab"],
                "langfuse_session_id": "tool-tuning-lab",
            },
        },
    )

    # Tool 호출 추출
    tool_calls = []
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tool_calls.extend(tc["name"] for tc in msg.tool_calls)

    final_answer = result["messages"][-1].content
    bad_results.append({"question": q, "tools_used": tool_calls, "answer": final_answer})
    print(f"  Q: {q}")
    print(f"     Tool 호출: {tool_calls or '없음'}")
    print(f"     응답: {final_answer[:100]}...\n")

🔴 나쁜 Tool 설명으로 실행:

  Q: 재택근무 신청은 어떻게 하나요?
     Tool 호출: 없음
     응답: 재택근무 신청 방법은 회사마다 조금씩 다르지만, 보통은 아래 순서로 진행합니다.

1. **사내 규정 확인**
   - 재택근무 가능 요일, 대상자, 신청 기한, 승인 기준을 먼저...

  Q: 김철수 대리 내선번호 알려주세요
     Tool 호출: ['search']
     응답: 죄송하지만, 현재 검색된 결과에는 내선번호 정보가 없습니다.  
원하시면 제가 다시 찾아보거나, 부서명/팀명까지 알려주시면 더 정확히 확인해볼게요....

  Q: 내일 오후 회의실 빈 시간 알려주세요
     Tool 호출: 없음
     응답: 회의실 일정을 확인하려면 회의실 캘린더/예약 시스템에 접근할 수 있어야 합니다.  
제가 지금 바로 확인해드리려면 아래 정보가 필요해요:

1. **어느 회의실인지**  
2. *...

  Q: 오늘 서울 날씨 어때요?
     Tool 호출: ['get_info']
     응답: 오늘 서울 날씨를 바로 확인하려 했는데, 현재 조회 결과가 날씨가 아니라 일정 정보로 나왔어요.

확인된 내용:
- 10시 팀회의
- 14시 회의실 A 예약

원하시면 제가 다시 ...



## 2. Tool 설명 개선

좋은 Tool 설명에는 3가지가 포함되어야 한다:
1. **언제 사용하는가** — 구체적인 사용 상황
2. **언제 사용하지 않는가** — 혼동하기 쉬운 상황 명시
3. **파라미터 예시** — 어떤 값을 넣어야 하는지

In [13]:
# ── 좋은 Tool 정의: 명확한 이름과 상세 설명 ──

@tool
def search_internal_docs(query: str) -> str:
    """사내 정책, 매뉴얼, 절차서를 키워드로 검색한다.

    사용: 회사 정책, 업무 프로세스, 규정에 대한 질문
    미사용: 특정 직원 정보 조회, 일정/회의실 확인, 일반 상식
    예시: "재택근무 신청 방법", "출장비 정산 절차"

    Args:
        query: 검색 키워드 (예: "재택근무 정책")
    """
    return f"[사내 문서] '{query}' 검색 결과: 재택근무는 주 3회 가능, 팀장 승인 필요"


@tool
def lookup_employee(name: str) -> str:
    """직원의 인사 정보(부서, 직급, 연락처)를 이름으로 조회한다.

    사용: 특정 직원의 소속 부서, 직급, 내선번호를 알아야 할 때
    미사용: 사내 정책 검색, 일정 조회, 조직도 전체 조회
    예시: name="김철수"

    Args:
        name: 직원 이름 (예: "김철수")
    """
    return f"[인사 정보] {name}: 개발팀 시니어 엔지니어, 내선 1234"


@tool
def check_schedule(date: str) -> str:
    """특정 날짜의 회의실 예약 현황과 팀 일정을 확인한다.

    사용: 회의실 빈 시간 확인, 특정 날짜 팀 일정 조회
    미사용: 사내 정책 검색, 직원 정보 조회, 날씨/뉴스
    예시: date="2026-03-26"

    Args:
        date: 날짜 (YYYY-MM-DD 형식, 예: "2026-03-26")
    """
    return f"[일정] {date}: 10시 팀회의, 14시 회의실A 예약됨, 나머지 시간 가능"


good_tools = [search_internal_docs, lookup_employee, check_schedule]
print("좋은 Tool 목록:")
for t in good_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

좋은 Tool 목록:
  - search_internal_docs: 사내 정책, 매뉴얼, 절차서를 키워드로 검색한다.

    사용: 회사 정책, 업무 프로세스, 규정에 대한 ...
  - lookup_employee: 직원의 인사 정보(부서, 직급, 연락처)를 이름으로 조회한다.

    사용: 특정 직원의 소속 부서, 직급...
  - check_schedule: 특정 날짜의 회의실 예약 현황과 팀 일정을 확인한다.

    사용: 회의실 빈 시간 확인, 특정 날짜 팀 ...


In [14]:
# ── 좋은 Tool로 동일 테스트 실행 ──
good_agent = create_agent(model=llm, tools=good_tools)
handler = CallbackHandler()

good_results = []
print("🟢 좋은 Tool 설명으로 실행:\n")
for q in test_questions:
    result = good_agent.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["good-tools", "tool-tuning-lab"],
                "langfuse_session_id": "tool-tuning-lab",
            },
        },
    )

    tool_calls = []
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tool_calls.extend(tc["name"] for tc in msg.tool_calls)

    final_answer = result["messages"][-1].content
    good_results.append({"question": q, "tools_used": tool_calls, "answer": final_answer})
    print(f"  Q: {q}")
    print(f"     Tool 호출: {tool_calls or '없음'}")
    print(f"     응답: {final_answer[:100]}...\n")

🟢 좋은 Tool 설명으로 실행:

  Q: 재택근무 신청은 어떻게 하나요?
     Tool 호출: ['search_internal_docs']
     응답: 재택근무는 **주 3회까지 가능**하며, **팀장 승인**이 필요합니다.

보통은 사내 신청서나 인사/업무 시스템에서 신청한 뒤 팀장 승인 절차를 거칩니다.  
원하시면 제가 **...

  Q: 김철수 대리 내선번호 알려주세요
     Tool 호출: ['lookup_employee']
     응답: 김철수 대리의 내선번호는 **1234**입니다....

  Q: 내일 오후 회의실 빈 시간 알려주세요
     Tool 호출: ['check_schedule']
     응답: 내일(2026-03-27) 회의실 빈 시간은 다음과 같습니다.

- 10시 팀회의
- 14시 회의실 A 예약됨

따라서 **그 외 시간은 가능**합니다.  
원하시면 제가 **오후...

  Q: 오늘 서울 날씨 어때요?
     Tool 호출: 없음
     응답: 죄송하지만 현재는 실시간 날씨 정보를 조회할 수 없어요.  
대신 원하시면 **서울의 오늘 날씨를 확인하는 방법**이나, **날씨 앱/웹에서 빠르게 보는 법**을 안내해드릴게요....



## 3. 개선 효과 비교

In [15]:
# ── 기대 Tool vs 실제 호출 비교 ──
expected_tools = [
    ["search_internal_docs"],   # 재택근무 → 사내 문서 검색
    ["lookup_employee"],        # 김철수 내선 → 인사 정보 조회
    ["check_schedule"],         # 회의실 → 일정 확인
    [],                         # 날씨 → Tool 불필요
]

print("📊 Tool 선택 정확도 비교")
print("=" * 70)
print(f"{'질문':<30} {'기대 Tool':<22} {'나쁜 설명':<15} {'좋은 설명':<15}")
print("-" * 70)

bad_correct = 0
good_correct = 0
for i, q in enumerate(test_questions):
    expected = expected_tools[i]
    bad_used = bad_results[i]["tools_used"]
    good_used = good_results[i]["tools_used"]

    bad_match = "✅" if bad_used == expected else "❌"
    good_match = "✅" if good_used == expected else "❌"

    if bad_used == expected:
        bad_correct += 1
    if good_used == expected:
        good_correct += 1

    exp_str = expected[0] if expected else "(없음)"
    bad_str = bad_used[0] if bad_used else "(없음)"
    good_str = good_used[0] if good_used else "(없음)"

    print(f"{q[:28]:<30} {exp_str:<22} {bad_match} {bad_str:<12} {good_match} {good_str}")

print(f"\n정확도: 나쁜 설명 {bad_correct}/{len(test_questions)}, 좋은 설명 {good_correct}/{len(test_questions)}")

📊 Tool 선택 정확도 비교
질문                             기대 Tool                나쁜 설명           좋은 설명          
----------------------------------------------------------------------
재택근무 신청은 어떻게 하나요?              search_internal_docs   ❌ (없음)         ✅ search_internal_docs
김철수 대리 내선번호 알려주세요              lookup_employee        ❌ search       ✅ lookup_employee
내일 오후 회의실 빈 시간 알려주세요           check_schedule         ❌ (없음)         ✅ check_schedule
오늘 서울 날씨 어때요?                  (없음)                   ❌ get_info     ✅ (없음)

정확도: 나쁜 설명 0/4, 좋은 설명 4/4


## 4. 파라미터 Validation 레이어

Tool 선택이 맞아도 파라미터가 잘못되면 실패한다.
Validation 레이어를 추가하여 **호출 전에 오류를 차단**한다.

In [16]:
import re
from datetime import datetime


def validate_date(date_str: str) -> str | None:
    """YYYY-MM-DD 형식인지 검증한다. 유효하면 None, 아니면 에러 메시지."""
    if not re.match(r"^\d{4}-\d{2}-\d{2}$", date_str):
        return f"날짜 형식 오류: '{date_str}' → YYYY-MM-DD 형식이어야 합니다"
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
        return None
    except ValueError:
        return f"유효하지 않은 날짜: '{date_str}'"


def validate_name(name: str) -> str | None:
    """이름이 비어있거나 숫자만 포함하는지 검증한다."""
    if not name or not name.strip():
        return "이름이 비어 있습니다"
    if name.strip().isdigit():
        return f"이름에 숫자만 포함됨: '{name}'"
    return None


# ── Validation 적용된 Tool ──
@tool
def check_schedule_validated(date: str) -> str:
    """특정 날짜의 회의실 예약 현황과 팀 일정을 확인한다.

    사용: 회의실 빈 시간 확인, 특정 날짜 팀 일정 조회
    미사용: 사내 정책 검색, 직원 정보 조회

    Args:
        date: 날짜 (YYYY-MM-DD 형식, 예: "2026-03-26")
    """
    error = validate_date(date)
    if error:
        return f"⚠️ 파라미터 오류: {error}"
    return f"[일정] {date}: 10시 팀회의, 14시 회의실A 예약됨, 나머지 시간 가능"


# 테스트
test_dates = ["2026-03-26", "내일", "2026-13-01", "03/26"]
print("📋 날짜 Validation 테스트:")
for d in test_dates:
    result = check_schedule_validated.invoke(d)
    print(f"  '{d}' → {result}")

📋 날짜 Validation 테스트:
  '2026-03-26' → [일정] 2026-03-26: 10시 팀회의, 14시 회의실A 예약됨, 나머지 시간 가능
  '내일' → ⚠️ 파라미터 오류: 날짜 형식 오류: '내일' → YYYY-MM-DD 형식이어야 합니다
  '2026-13-01' → ⚠️ 파라미터 오류: 유효하지 않은 날짜: '2026-13-01'
  '03/26' → ⚠️ 파라미터 오류: 날짜 형식 오류: '03/26' → YYYY-MM-DD 형식이어야 합니다


## 5. Langfuse에서 확인할 포인트

1. **Tags 필터**: `bad-tools` vs `good-tools`로 필터링하여 Tool 호출 차이 비교
2. **Trace 상세**: 각 트레이스에서 LLM이 어떤 Tool을 선택했는지, 파라미터는 무엇인지 확인
3. **Sessions → `tool-tuning-lab`**: 전체 실험을 세션으로 묶어 한눈에 비교
4. **Tool 호출 실패 패턴**: Tool 호출 후 에러가 발생한 트레이스 식별
5. **Token Usage**: 좋은 Tool 설명은 description이 길어 입력 토큰이 늘지만, 재시도가 줄어 전체 비용은 오히려 감소하는지 확인

---

## 🔧 실습: 새로운 Tool 추가 및 정확도 테스트

**목표**: 아래 2개 Tool을 **좋은 설명**으로 정의하고, 테스트 질문을 추가하여 정확도를 측정한다.

추가할 Tool:
1. `book_meeting_room` — 회의실 예약
2. `submit_expense` — 경비 청구

In [17]:
# TODO: 좋은 설명으로 Tool을 정의하세요

@tool
def book_meeting_room(room: str, date: str, start_time: str, end_time: str) -> str:
    """TODO: 특정 날짜의 회의실 예약 현황과 팀 일정을 확인한다.

    사용: 회의실 빈 시간 확인, 특정 날짜 팀 일정 조회
    미사용: 사내 정책 검색, 직원 정보 조회, 날씨/뉴스
    예시: date="2026-03-26"

    Args:
        room: 회의실 이름 (예: "회의실A")
        date: 날짜 (YYYY-MM-DD)
        start_time: 시작 시간 (HH:MM)
        end_time: 종료 시간 (HH:MM)
    """
    return f"✅ {date} {start_time}~{end_time} {room} 예약 완료"


@tool
def submit_expense(amount: int, category: str, description: str) -> str:
    """TODO: 법인카드 경비 청구 방안에 대한 가이드를 제공한다.

    사용: 법인카드 월 사용 한도, 법인 카드 청구 유형 분류, 법인 카드 청구시 필수 작성 정보 안내
    미사용: 개인 카드 사용, 현금 결제
    예시: amount=50000, category="식비", description="점심 회식"

    Args:
        amount: 금액 (예 : 500,000원)
        category: 분류 ("교통비", "식비", "숙박비", "교육비")
        description: 법인카드 월 사용 한도는 300만원이며, 청구 시에는 사용 내역과 영수증 첨부가 필요합니다. (예: "점심 회식")
                     법인카드 분류 유형은 "교통비", "식비", "숙박비", "교육비" 등이 있으며, 청구 시에는 해당 분류와 금액, 사용 내역을 정확히 작성해야 합니다.
                     법인카드 청구시 필수 작성 정보에는 금액, 분류, 사용 내역 설명이 포함됩니다. (예: amount=50000, category="식비", description="점심 회식")
    """
    return f"✅ 법인카드 청구 접수: {category} {amount:,}원 - {description}"


# TODO: 테스트 질문 추가
new_test_questions = [
    "내일 14시부터 15시까지 회의실A 예약해줘",  # → book_meeting_room
    "점심 회식비 5만원을 식비로 경비 청구해줘",   # → submit_expense
    "출장비 정산 절차 알려줘",                     # → search_internal_docs
    "오늘 부산 날씨 어때?",                        # → Tool 불필요
]

# TODO: 에이전트 생성 + 실행
all_tools = [
    search_internal_docs,
    lookup_employee,
    check_schedule,
    book_meeting_room,
    submit_expense,
 ]
agent = create_agent(model=llm, tools=all_tools)
handler = CallbackHandler()

print("🧪 확장 Tool 세트 테스트 실행\n")
extended_results = []
for q in new_test_questions:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["extended-tools", "tool-tuning-lab"],
                "langfuse_session_id": "tool-tuning-lab",
            },
        },
    )

    tool_calls = []
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tool_calls.extend(tc["name"] for tc in msg.tool_calls)

    final_answer = result["messages"][-1].content
    extended_results.append({"question": q, "tools_used": tool_calls, "answer": final_answer})

    print(f"Q: {q}")
    print(f"   Tool 호출: {tool_calls or '없음'}")
    print(f"   응답: {final_answer[:120]}...\n")

# 간단 정확도 측정
expected_tools_new = [
    ["book_meeting_room"],
    ["submit_expense"],
    ["search_internal_docs"],
    [],
]

correct = 0
for i, row in enumerate(extended_results):
    if row["tools_used"] == expected_tools_new[i]:
        correct += 1

print(f"확장 테스트 정확도: {correct}/{len(new_test_questions)}")

🧪 확장 Tool 세트 테스트 실행

Q: 내일 14시부터 15시까지 회의실A 예약해줘
   Tool 호출: ['check_schedule', 'book_meeting_room']
   응답: 예약 완료했습니다.

- 회의실: 회의실A
- 일시: 2026-03-27 14:00~15:00

참고로 내일 14시에 이미 회의실A 예약이 있었는데, 요청하신 시간대로 최종 예약이 완료되었습니다....

Q: 점심 회식비 5만원을 식비로 경비 청구해줘
   Tool 호출: ['submit_expense']
   응답: ✅ 경비 청구가 접수되었습니다.

- 분류: 식비
- 금액: 50,000원
- 내역: 점심 회식...

Q: 출장비 정산 절차 알려줘
   Tool 호출: ['search_internal_docs']
   응답: 검색 결과가 출장비 정산 절차와 맞지 않고, 재택근무 관련 문서만 확인되었습니다.

원하시면 제가 다시 다른 키워드로 찾아드릴게요. 예를 들면:
- 출장비
- 출장 정산
- 여비 규정
- 경비 청구

원하시는 방향이...

Q: 오늘 부산 날씨 어때?
   Tool 호출: 없음
   응답: 죄송하지만 현재 저는 실시간 날씨를 직접 조회할 수는 없어요.

원하시면 제가 대신:
- **부산 오늘 날씨를 확인하는 방법**을 알려드리거나
- **날씨 앱/웹에서 바로 확인할 수 있는 검색어**를 드릴게요.

예...

확장 테스트 정확도: 3/4
